# Server GPU cho nghiên cứu TEMPORAL (Video LM hiểu thời gian; SVHalluc TA / TF / CMTB): nạp Qwen2.5-Omni-3B một lần, nhận job qua ngrok

**Cách dùng** (chi tiết từng bước trong `README.md` của repo `Temporal_Hallucinate_VideoLLM`):
1. **Add-ons → Secrets → Add Secret**: Label `NGROK_AUTHTOKEN`, Value = authtoken ngrok của bạn, và **bật Attach** cho notebook này.
2. Session options: **GPU T4 ×2**, **Internet On**. Rồi **Save Version → Save & Run All (Commit)**.
3. Mở Logs của bản đang chạy. Sau khoảng 5–10 phút, log in ra **URL** (`https://….ngrok-free.dev`) và **TOKEN**. Ghi hai giá trị vào `remote_config.json` ở máy bạn (file này bị `.gitignore` chặn, **không bao giờ commit**).
4. Server sống tới hết phiên (tối đa 12 giờ). `python remote_client.py restart` khởi động lại server nếu job treo.

**Bảo mật.** Ai có cả URL lẫn TOKEN đều chạy được mã tuỳ ý trong notebook này. Không đăng log, không gửi URL/TOKEN vào kênh công khai. Đặt `ALLOW_EXEC = False` ở cell cuối nếu chỉ muốn thử kết nối. Dùng tài khoản Kaggle phụ; notebook cố ý không có vòng lặp chống ngắt.

In [ ]:
# CELL 0 — kiểm tra Internet
import socket
for host in ('pypi.org', 'huggingface.co', 'ngrok.com'):
    try:
        socket.create_connection((host, 443), timeout=8).close(); print('OK  ', host)
    except OSError as e:
        raise RuntimeError('Không kết nối được ' + host + '. Bật Internet: Session options -> Internet = On.') from e


In [ ]:
# CELL 1 — môi trường: venv riêng với các phiên bản đã ghim (transformers 4.57.6 cho Qwen2.5-Omni), giữ nguyên torch của Kaggle
import os, sys, json, subprocess, importlib.metadata, venv, time
from pathlib import Path
assert Path('/kaggle').exists(), 'Chạy trên Kaggle (GPU T4 x2, Internet On).'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'; os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True'); os.environ['FORCE_QWENVL_VIDEO_READER'] = 'decord'
ROOT = Path('/kaggle/working/temporal_lab_runner'); ROOT.mkdir(parents=True, exist_ok=True)
OUT = Path('/kaggle/working/temporal_lab_' + time.strftime('%Y%m%d_%H%M%S')); OUT.mkdir(parents=True, exist_ok=True)
ENV = ROOT / 'env'
venv.EnvBuilder(system_site_packages=True, with_pip=False).create(ENV)          # no ensurepip; the host pip installs into it
PYTHON = ENV / 'bin' / 'python'
PIP = [sys.executable, '-m', 'pip', '--python', str(PYTHON)]
keep = []
for p in ('torch', 'torchvision', 'torchaudio'):
    try: keep.append(p + '==' + importlib.metadata.version(p))
    except importlib.metadata.PackageNotFoundError: pass
(ROOT / 'keep_torch.txt').write_text(chr(10).join(keep) + chr(10), encoding='utf-8')
packages = ['transformers==4.57.6', 'accelerate==1.12.0', 'huggingface-hub==0.36.0', 'qwen-omni-utils==0.0.8', 'decord==0.6.0', 'av==16.0.1',
            'librosa==0.11.0', 'soundfile>=0.12,<0.14', 'numpy>=2,<2.4', 'Pillow>=10,<13', 'sentencepiece>=0.2,<0.3', 'protobuf>=5,<7']
with (OUT / 'install.log').open('a', encoding='utf-8') as log:
    proc = subprocess.Popen([*PIP, 'install', '--only-binary=:all:', '--constraint', str(ROOT / 'keep_torch.txt'), *packages],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    for line in proc.stdout: print(line, end='', flush=True); log.write(line)
    if proc.wait(): raise RuntimeError('Cài thư viện thất bại; xem ' + str(OUT / 'install.log'))
subprocess.run([str(PYTHON), '-c', 'import torch, av, decord; from transformers import Qwen2_5OmniThinkerForConditionalGeneration; '
                'print("torch", torch.__version__, "| GPUs", torch.cuda.device_count(), "| av", av.__version__)'], check=True)
print('Môi trường sẵn sàng. Kết quả và log:', OUT)


In [ ]:
# CELL 2 — ghi mã nguồn server
SERVER_FILES = {'temporal_lab.py': '"""Shared lab namespace for TEMPORAL research on Qwen2.5-Omni-3B (Thinker): does the video LM really understand time?\nDefault data: SVHalluc temporal tasks TA / TF / CMTB (983 questions).\n\nThe GPU server (server/kaggle_gpu_server.py) executes this file ONCE, loads the model, and then runs every job snippet in the\nsame namespace, so a job can directly use:\n    MODEL_, PROC, L4                       model, processor, token ids of the letters A B C D\n    fetch(name)                            file of the SVHalluc dataset (HF, pinned revision) -> local Path\n    temporal_rows()                        all temporal rows of samples.json as (idx, row)\n    clip_info(row)                         (path, duration_s, audio_codec)\n    balanced_items(per_cell, seed, ...)    balanced per (task, gold letter), round robin, unusable clips skipped BEFORE selection\n    parse_question(q) -> (stem, options)   prompt_of(stem, options, perm)   rotations(k)\n    prepare_video(path)                    Qwen-official preprocessing (video via decord, audio via PyAV)\n    qwen_inputs(PROC, prompt, media, mode) mode \'av\' | \'vision\' | \'audio\';  move_inputs(inputs, MODEL_)\n    score(media, stem, options, mode, fwd, both)\n                                           log-prob of every option summed over all rotations of the option order;\n                                           both=True also returns the log-probs in the ORIGINAL option order only\n                                           (the protocol of published tables such as RED\'s), from the same forwards\n    media_layout(MODEL_, PROC, inputs)     rope position ids, seconds per token, video / audio token masks (for interventions)\n    narrated_block(audio)                  (start_s, end_s) of the narrated block of an SVHalluc clip (analysis only)\n    run_loop(items, out_name, fn, show)    resumable loop (by idx) writing OUT/<out_name>, stops cleanly before the deadline\n    OUT                                    results folder (download files with: python remote_client.py get <rel> <local>)\n\nDATA FACTS (measured on the local copy, 2026-10-20):\n  * temporal clips: 983 questions (TA 368, TF 276, CMTB 339), every clip has AAC audio (164/164 checked): no codec/label leak\n    (unlike SVHalluc semantic, where gold-No clips are pcm and gold-Yes clips AAC).\n  * the audio of a clip is digital silence + ONE continuous narrated block with whole-second edges (median ~13.6 s of a 23-28 s clip).\n  * TA gold A/B balanced (184/184), TF A/B/C balanced (92 each), CMTB A/B/C balanced (113 each).\nLabels are used only for scoring and analysis; nothing here feeds a label to the model."""\nimport os, sys, json, re, time, random\nfrom pathlib import Path\nimport numpy as np\nimport torch\n\nos.environ.setdefault(\'FORCE_QWENVL_VIDEO_READER\', \'decord\')\nROOT = Path(os.environ.get(\'LAB_ROOT\', Path(globals().get(\'__file__\', \'temporal_lab.py\')).resolve().parent))\nOUT = Path(os.environ.get(\'LAB_OUT\', ROOT / \'out\')); OUT.mkdir(parents=True, exist_ok=True)\nif str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))\nimport av\nfrom huggingface_hub import hf_hub_download\n\nDATASET, REVISION = \'zcs15/SVHalluc\', \'211b4f14ab66ae9557b69e302d1f090e0e610839\'\nMODEL, MREV = \'Qwen/Qwen2.5-Omni-3B\', \'f75b40e3da2003cdd6e1829b1f420ca70797c34e\'\nSR = 16000; BS = chr(92)\nTEMPORAL_TASKS = (\'TA\', \'TF\', \'CMTB\')\nNEUTRAL = \'What is happening in the video?\'\nMAX_DUR = float(os.environ.get(\'LAB_MAX_DUR\', 42.0 if torch.cuda.device_count() >= 2 else 28.0))\nDEADLINE = time.time() + float(os.environ.get(\'LAB_HOURS\', 11.5)) * 3600\n\n\n# ---------------------------------------------------------------- audio (PyAV only; no system ffmpeg is needed to read the clips)\ndef decode_audio(path):\n    chunks = []\n    with av.open(str(path)) as c:\n        rs = av.AudioResampler(format=\'fltp\', layout=\'mono\', rate=SR)\n        for fr in c.decode(audio=0):\n            for p in rs.resample(fr): chunks.append(p.to_ndarray().reshape(-1))\n        for p in rs.resample(None): chunks.append(p.to_ndarray().reshape(-1))\n    return np.concatenate(chunks).astype(\'float32\', copy=False)\n\n\ndef narrated_block(a):\n    """SVHalluc audio = digital silence + one continuous narrated block: (start_s, end_s), or None. Analysis only."""\n    hop = 800; n = len(a) // hop\n    db = 20 * np.log10(np.sqrt((a[:n * hop].reshape(n, hop) ** 2).mean(1)) + 1e-9)\n    on = np.where(db > -80)[0]\n    return None if len(on) == 0 else (float(on[0] * 0.05), float((on[-1] + 1) * 0.05))\n\n\ntry:                                       # qwen-omni-utils reads the audio of an mp4 through librosa.load -> route it to PyAV\n    import librosa\n    _orig_load = librosa.load\n    def _load(path, sr=None, offset=0.0, duration=None, **k):\n        if str(path).lower().endswith(\'.mp4\'):\n            a = decode_audio(path)\n            return a[int(sr * offset): None if duration is None else int(sr * (offset + duration))], sr\n        return _orig_load(path, sr=sr, offset=offset, duration=duration, **k)\n    librosa.load = _load\nexcept ImportError:\n    pass\n\nfrom lab_native_video import prepare_video, qwen_inputs, move_inputs, SYSTEM\nfrom speech_gate import media_layout\n\n\n# ---------------------------------------------------------------- data\ndef fetch(name):\n    local = os.environ.get(\'LAB_LOCAL_DATA\')\n    if local and (Path(local) / name).exists(): return Path(local) / name\n    return Path(hf_hub_download(DATASET, name, repo_type=\'dataset\', revision=REVISION))\n\n\ndef temporal_rows():\n    rows = json.loads(fetch(\'samples.json\').read_text(encoding=\'utf-8\'))\n    return [(i, r) for i, r in enumerate(rows) if r[\'data_type\'] == \'temporal\']\n\n\ndef clip_info(r):\n    path = fetch(r[\'video_path\'].replace(BS, \'/\'))\n    with av.open(str(path)) as c:\n        dur = float(c.duration / 1e6) if c.duration else None\n        codec = c.streams.audio[0].codec_context.name if c.streams.audio else \'none\'\n    return path, dur, codec\n\n\ndef parse_question(q):\n    parts = re.split(r\'\\s*\\(([ABCD])\\)\\s*\', q)\n    return parts[0].strip(), [parts[i + 1].strip() for i in range(1, len(parts) - 1, 2)]\n\n\nSKIPPED = []\n\n\ndef usable(r, max_dur=None):\n    try:\n        _, dur, codec = clip_info(r)\n        if codec == \'none\': raise RuntimeError(\'no audio stream\')\n        if dur is None or dur > (max_dur or MAX_DUR): raise RuntimeError(\'long\')\n        return True\n    except Exception as e:\n        SKIPPED.append((r[\'task_type_abbr\'], r[\'answer\'], repr(e)[:60])); return False\n\n\ndef balanced_items(per_cell, seed, tasks=TEMPORAL_TASKS, max_dur=None, exclude=()):\n    """per_cell items per (task, gold letter), shuffled with the seed, unusable clips skipped BEFORE selection, round robin."""\n    cells = {}\n    for i, r in temporal_rows():\n        if r[\'task_type_abbr\'] in tasks and i not in set(exclude): cells.setdefault((r[\'task_type_abbr\'], r[\'answer\']), []).append((i, r))\n    pools = []\n    for key in sorted(cells):\n        pool = cells[key][:]; random.Random(f\'{seed}-{key}\').shuffle(pool); got = []\n        for i, r in pool:\n            if usable(r, max_dur): got.append((i, r))\n            if len(got) == per_cell: break\n        pools.append(got)\n    out = []\n    for k in range(max((len(p) for p in pools), default=0)):\n        for p in pools:\n            if k < len(p): out.append(p[k])\n    return out\n\n\n# ---------------------------------------------------------------- model and scoring\nMODEL_ = PROC = L4 = None\n\n\ndef load_model():\n    from transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor\n    cache = os.environ.get(\'LAB_CACHE\')\n    kw = dict(revision=MREV, torch_dtype=torch.float16, attn_implementation=\'sdpa\', cache_dir=cache)\n    if torch.cuda.device_count() >= 2: kw.update(device_map=\'balanced\', max_memory={0: \'10GiB\', 1: \'10GiB\'})\n    else: kw.update(device_map={\'\': 0})\n    m = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(MODEL, **kw).eval()\n    for p in m.parameters(): p.requires_grad_(False)\n    return m, Qwen2_5OmniProcessor.from_pretrained(MODEL, revision=MREV, cache_dir=cache)\n\n\ndef rotations(k): return [tuple((s + j) % k for j in range(k)) for s in range(k)]\ndef prompt_of(stem, texts, perm): return stem + \' \' + \' \'.join(f\'({L}) {texts[j]}\' for L, j in zip(\'ABCD\', perm))\n\n\ndef last_logits(inp, mode=\'av\'):\n    """plain forward; lm_head only on the last position (saves memory / time)."""\n    hk = MODEL_.lm_head.register_forward_pre_hook(lambda mod, args: (args[0][:, -1:, :],))\n    try:\n        with torch.no_grad(): return MODEL_(**inp, use_audio_in_video=(mode == \'av\')).logits[0, -1].float()\n    finally: hk.remove()\n\n\ndef score(media, stem, texts, mode=\'av\', fwd=None, both=False):\n    """log-prob of each option letter, summed over every rotation of the option order (removes letter-position bias).\n    fwd(inputs) -> last-token logits replaces the plain forward (interventions).\n    both=True returns (rotation_sum, canonical): canonical = log-probs with the options in their ORIGINAL order only\n    (rotation 0), i.e. the protocol of tables that report plain accuracy; it comes from the same forwards."""\n    k = len(texts); tot = np.zeros(k); canon = None\n    for r_i, perm in enumerate(rotations(k)):\n        inp, _ = qwen_inputs(PROC, prompt_of(stem, texts, perm), media, mode); inp = move_inputs(inp, MODEL_)\n        lg = last_logits(inp, mode) if fwd is None else fwd(inp)\n        lp = torch.log_softmax(lg[L4[:k]].cpu(), -1).numpy()\n        if r_i == 0: canon = lp.round(3).tolist()                       # rotation 0 = identity order\n        for p_, j in enumerate(perm): tot[j] += lp[p_]\n        del inp\n    torch.cuda.empty_cache()\n    rs = tot.round(3).tolist()\n    return (rs, canon) if both else rs\n\n\n# ---------------------------------------------------------------- resumable loop\ndef time_left(): return DEADLINE - time.time()\n\n\ndef run_loop(items, out_name, fn, show=(), min_left=900):\n    """fn(idx, row) -> dict.  One JSON line per item in OUT/out_name; items already in the file are skipped (resume).\n    show: keys of score lists to print correct/wrong per (task, gold) as the run goes."""\n    res = OUT / out_name\n    done = {json.loads(l)[\'idx\'] for l in res.read_text(encoding=\'utf-8\').splitlines() if l.strip()} if res.exists() else set()\n    print(\'items\', len(items), \'already done\', len(done), flush=True)\n    stats, out = {}, []\n    for n, (i, r) in enumerate(items):\n        if i in done: continue\n        if time_left() < min_left: print(\'deadline reached: stopping cleanly\', flush=True); break\n        t0 = time.time(); rec = dict(idx=i, task=r[\'task_type_abbr\'], gold=\'ABCD\'.index(r[\'answer\']))\n        try: rec.update(fn(i, r)); rec[\'sec\'] = round(time.time() - t0)\n        except torch.cuda.OutOfMemoryError: rec[\'error\'] = \'OOM\'\n        except Exception as e: rec[\'error\'] = repr(e)[:300]\n        torch.cuda.empty_cache()\n        with res.open(\'a\', encoding=\'utf-8\') as f: f.write(json.dumps(rec) + \'\\n\')\n        out.append(rec)\n        if \'error\' in rec: print(n + 1, len(items), rec[\'task\'], i, \'ERR\', rec[\'error\'][:100], flush=True); continue\n        g = rec[\'gold\']; key = (rec[\'task\'], \'ABC\'[g])\n        for c in show:\n            if isinstance(rec.get(c), list): z = stats.setdefault((key, c), [0, 0]); z[int(np.argmax(rec[c])) != g] += 1\n        marks = \' \'.join(f\'{c}={"Y" if int(np.argmax(rec[c])) == g else "-"}\' for c in show if isinstance(rec.get(c), list))\n        print(n + 1, len(items), rec[\'task\'], i, marks, rec[\'sec\'], \'s\', flush=True)\n    if stats:\n        print(\'correct/wrong per (task, gold):\', {f\'{k[0][0]}/{k[0][1]} {k[1]}\': f\'{v[0]}/{v[1]}\' for k, v in sorted(stats.items())}, flush=True)\n    return out\n', 'lab_native_video.py': '"""Native video-file input through Qwen\'s official utilities; no evidence extraction."""\nfrom pathlib import Path\n\nSYSTEM=\'You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech.\'\n# T4 SDPA cannot hold the default ~18k-token attention matrix. Keep Qwen\'s\n# temporal sampling and original MP4 path, while bounding only pixels per frame.\nVIDEO_MAX_PIXELS=160*28*28\n\n\ndef conversation_for(path, text, media_mode=\'av\', max_pixels=VIDEO_MAX_PIXELS):\n    if media_mode not in (\'av\',\'vision\',\'audio\'):raise ValueError(\'Unknown media mode\')\n    media=({\'type\':\'audio\',\'audio\':str(path)} if media_mode==\'audio\' else\n           {\'type\':\'video\',\'video\':str(path),\'max_pixels\':max_pixels})\n    return [\n        {\'role\':\'system\',\'content\':[{\'type\':\'text\',\'text\':SYSTEM}]},\n        {\'role\':\'user\',\'content\':[media,\n                                  {\'type\':\'text\',\'text\':text}]},\n    ]\n\n\ndef prepare_video(path, max_pixels=VIDEO_MAX_PIXELS):\n    from qwen_omni_utils import process_mm_info\n    path=Path(path).resolve()\n    if not path.is_file():raise FileNotFoundError(path)\n    # MP4 path, not caller-extracted frames. Keep Qwen\'s default temporal sampling;\n    # max_pixels per frame only bounds resolution for the T4 memory budget.\n    # The notebook selects Qwen\'s supported Decord reader because Kaggle\'s\n    # torchvision build no longer provides torchvision.io.read_video.\n    audios,images,videos,video_kwargs=process_mm_info(\n        conversation_for(path,\'\',max_pixels=max_pixels),use_audio_in_video=True,return_video_kwargs=True)\n    if images is not None:raise RuntimeError(\'Unexpected supplemental images\')\n    if audios is None or len(audios)!=1 or videos is None or len(videos)!=1:\n        raise RuntimeError(\'Expected one original video and its audio\')\n    # Transformers 4.57.6 Omni expects a scalar fps for this single video.\n    fps=video_kwargs[\'fps\']\n    if isinstance(fps,(list,tuple)):\n        if len(fps)!=1:raise RuntimeError(\'Expected one video fps\')\n        fps=fps[0]\n    return {\'path\':str(path),\'audio\':audios,\'videos\':videos,\'fps\':float(fps),\n            \'max_pixels_per_frame\':max_pixels}\n\n\ndef qwen_inputs(processor, prompt, media, media_mode=\'av\'):\n    if media_mode not in (\'av\',\'vision\',\'audio\'):raise ValueError(\'Unknown media mode\')\n    use_audio=media_mode in (\'av\',\'audio\')\n    use_video=media_mode in (\'av\',\'vision\')\n    rendered=processor.apply_chat_template(conversation_for(media[\'path\'],prompt,media_mode,media[\'max_pixels_per_frame\']),\n        add_generation_prompt=True,tokenize=False)\n    inputs=processor(text=rendered,audio=media[\'audio\'] if use_audio else None,images=None,videos=media[\'videos\'] if use_video else None,\n        return_tensors=\'pt\',padding=True,\n        videos_kwargs={\'fps\':media[\'fps\'],\'use_audio_in_video\':media_mode==\'av\'})\n    if (\'input_features\' in inputs)!=use_audio or (\'pixel_values_videos\' in inputs)!=use_video:\n        raise RuntimeError(\'Native inputs do not match the requested modalities\')\n    if \'pixel_values\' in inputs:raise RuntimeError(\'Unexpected supplemental image tokens\')\n    # HF Transformers takes this switch from videos_kwargs, not its top-level\n    # kwargs. Check that the resulting token sequence actually contains audio.\n    audio_token_id=processor.tokenizer.convert_tokens_to_ids(processor.audio_token)\n    audio_tokens=int((inputs[\'input_ids\']==audio_token_id).sum().item())\n    expected_audio_tokens=0\n    if use_audio:\n        if \'feature_attention_mask\' not in inputs:\n            raise RuntimeError(\'Audio feature mask missing; cannot verify audio-token alignment\')\n        input_lengths=(inputs[\'feature_attention_mask\'].sum(-1)-1)//2+1\n        expected_audio_tokens=int((((input_lengths-2)//2+1).clamp_min(0)).sum().item())\n    if audio_tokens!=expected_audio_tokens or (use_audio and audio_tokens==0):\n        raise RuntimeError(f\'Audio tokens missing/misaligned: found {audio_tokens}, expected {expected_audio_tokens}\')\n    expected_video_tokens=0\n    if use_video:\n        grid=inputs[\'video_grid_thw\']\n        merge=processor.video_processor.merge_size**2\n        expected_video_tokens=int((grid.prod(-1).sum()//merge).item())\n    video_token_id=processor.tokenizer.convert_tokens_to_ids(processor.video_token)\n    video_tokens=int((inputs[\'input_ids\']==video_token_id).sum().item())\n    if video_tokens!=expected_video_tokens or (use_video and video_tokens==0):\n        raise RuntimeError(f\'Video tokens missing/misaligned: found {video_tokens}, expected {expected_video_tokens}\')\n    return inputs,rendered\n\n\ndef move_inputs(inputs, model):\n    import torch\n    inputs=inputs.to(model.get_input_embeddings().weight.device)\n    for key in (\'input_features\',\'pixel_values_videos\'):\n        if key in inputs:inputs[key]=inputs[key].to(torch.float16)\n    return inputs\n', 'speech_gate.py': '"""Speech-anchored attention gating for Qwen2.5-Omni Thinker (single forward pass, no training, no re-encoding).\n\nQwen2.5-Omni gives every audio and video token a temporal rope id (25 ids per second, shared clock).\nGiven a time window [s0, s1] (seconds), we find the video tokens whose temporal patch overlaps the window and,\nfor every text query position AFTER the media block (question + assistant prefix), we add a bias to the attention\nlogits of video keys OUTSIDE the window:  hard = -inf (attention-level crop), soft = -beta.\nMedia tokens themselves attend as usual, so the visual and audio representations are unchanged; only what the\nquestion/answer positions read from is steered.\n"""\nimport torch\n\n\ndef media_layout(model, proc, inputs):\n    """Return position_ids (3,1,L), per-token time in seconds for video tokens, and index sets."""\n    ids = inputs[\'input_ids\']\n    am = inputs[\'attention_mask\']\n    afl = inputs[\'feature_attention_mask\'].sum(-1) if \'feature_attention_mask\' in inputs else None\n    spg = inputs.get(\'video_second_per_grid\')\n    pos, _ = model.get_rope_index(ids, None, inputs.get(\'video_grid_thw\'), am, True, afl, spg)\n    tok = proc.tokenizer\n    vid_id = tok.convert_tokens_to_ids(proc.video_token)\n    aud_id = tok.convert_tokens_to_ids(proc.audio_token)\n    is_v = (ids[0] == vid_id)\n    is_a = (ids[0] == aud_id)\n    t = pos[0, 0].float()\n    base = t[is_a].min() if is_a.any() else t[is_v].min()\n    sec = (t - base) / 25.0                       # temporal rope id -> seconds on the shared clock\n    last_media = int(torch.nonzero(is_v | is_a).max())\n    return pos, sec, is_v, is_a, last_media\n\n\ndef gated_mask(L, is_v, sec, last_media, window, mode, beta, dtype, device, spg=None, scope=\'all\', is_a=None, block_audio=False):\n    """scope=\'text\': only positions after the media block are gated (leaks through media tokens that already read the\n    out-of-window frames). scope=\'all\': no position except the out-of-window video tokens themselves may read them,\n    i.e. an attention-level crop that keeps positions, audio and the full token sequence."""\n    neg = torch.finfo(dtype).min\n    m = torch.zeros(L, L, dtype=torch.float32, device=device)\n    m = m.masked_fill(torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1), float(\'-inf\'))\n    if mode != \'none\':\n        s0, s1 = window\n        width = float(spg) if spg is not None else 0.0   # a temporal patch covers [t, t + second_per_grid)\n        inside = (sec + width > s0) & (sec < s1)\n        out_v = is_v.to(device) & ~inside.to(device)\n        rows = (torch.arange(L, device=device) > last_media) if scope == \'text\' else ~out_v\n        bias = float(\'-inf\') if mode == \'hard\' else -float(beta)\n        m[rows.unsqueeze(1) & out_v.unsqueeze(0)] += bias\n    if block_audio:\n        # audio-blind decision: no non-audio position may read audio tokens (audio tokens still read each other)\n        aud = is_a.to(device)\n        m[(~aud).unsqueeze(1) & aud.unsqueeze(0)] = float(\'-inf\')\n    m = m.clamp(min=neg).to(dtype)\n    return m.view(1, 1, L, L)\n\n\ndef binding_mask(L, is_v, is_a, sec, dtype, device, chunk_s=2.0, delta=0, restrict=True, bidir=False, base=None):\n    """Synchronous audio-visual binding inside the media block (text rows are never changed).\n    Qwen2.5-Omni interleaves every 2-s chunk as [video_c, audio_c] under a causal mask, so\n      - video_c can never read audio_c (the sound of the SAME moment comes later in the sequence), and\n      - audio_c (and video_c) can read the other modality of ALL earlier moments.\n    restrict=True : a cross-modal edge (audio<->video) is kept only if the two chunks differ by <= delta\n    bidir=True    : additionally let video_c read audio_c\' for |c - c\'| <= delta even though it comes later (prefill only)\n    Within-modality attention and everything involving text tokens keep the original causal pattern."""\n    m = torch.zeros(L, L, dtype=torch.float32, device=device) if base is None else base.clone().float()\n    if base is None:\n        m = m.masked_fill(torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1), float(\'-inf\'))\n    v, a = is_v.to(device), is_a.to(device)\n    ch = torch.floor(sec.to(device) / chunk_s)\n    near = (ch.unsqueeze(1) - ch.unsqueeze(0)).abs() <= delta\n    cross = (v.unsqueeze(1) & a.unsqueeze(0)) | (a.unsqueeze(1) & v.unsqueeze(0))\n    if restrict:\n        m[cross & ~near] = float(\'-inf\')\n    if bidir:\n        later = torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1)\n        m[(v.unsqueeze(1) & a.unsqueeze(0)) & near & later] = 0.0\n    neg = torch.finfo(dtype).min\n    return m.clamp(min=neg).to(dtype).view(1, 1, L, L)\n\n\ndef balance_mask(L, is_v, is_a, last_media, dtype, device, strength=1.0):\n    """Modality-balanced readout: question/answer rows get an additive bias on audio keys equal to\n    strength * log(n_video / n_audio), so that with equal content scores the two streams receive equal total attention\n    (Qwen2.5-Omni gives ~144 video tokens vs ~25 audio tokens per second)."""\n    m = torch.zeros(L, L, dtype=torch.float32, device=device)\n    m = m.masked_fill(torch.triu(torch.ones(L, L, dtype=torch.bool, device=device), 1), float(\'-inf\'))\n    v, a = is_v.to(device), is_a.to(device)\n    b = strength * float(torch.log(v.sum().float() / a.sum().float().clamp(min=1)))\n    rows = torch.arange(L, device=device) > last_media\n    m[rows.unsqueeze(1) & a.unsqueeze(0)] += b\n    return m.clamp(min=torch.finfo(dtype).min).to(dtype).view(1, 1, L, L)\n\n\ndef bound_last_logits(model, proc, inputs, delta=0, restrict=True, bidir=False):\n    pos, sec, is_v, is_a, last_media = media_layout(model, proc, inputs)\n    L = inputs[\'input_ids\'].shape[1]\n    emb = model.get_input_embeddings().weight\n    mask = binding_mask(L, is_v, is_a, sec, emb.dtype, emb.device, delta=delta, restrict=restrict, bidir=bidir)\n    kw = {k: v for k, v in inputs.items() if k != \'attention_mask\'}\n    with torch.no_grad():\n        out = model(**kw, attention_mask=mask, position_ids=pos.to(emb.device), use_audio_in_video=True)\n    return out.logits[0, -1].float()\n\n\ndef gated_last_logits(model, proc, inputs, window=None, mode=\'none\', beta=0.0, use_audio_in_video=True, scope=\'all\', block_audio=False):\n    """Forward pass with explicit 3D rope ids and a 4D additive mask; returns last-position logits (float32)."""\n    pos, sec, is_v, is_a, last_media = media_layout(model, proc, inputs)\n    L = inputs[\'input_ids\'].shape[1]\n    emb = model.get_input_embeddings().weight\n    spg = inputs.get(\'video_second_per_grid\')\n    spg = float(spg.reshape(-1)[0]) if spg is not None else None\n    mask = gated_mask(L, is_v, sec, last_media, window, mode, beta, emb.dtype, emb.device, spg, scope, is_a, block_audio)\n    kw = {k: v for k, v in inputs.items() if k != \'attention_mask\'}\n    with torch.no_grad():\n        out = model(**kw, attention_mask=mask, position_ids=pos.to(emb.device), use_audio_in_video=use_audio_in_video)\n    return out.logits[0, -1].float(), dict(n_video=int(is_v.sum()), n_audio=int(is_a.sum()), last_media=last_media,\n                                           video_sec_range=[float(sec[is_v].min()), float(sec[is_v].max())])\n', 'kaggle_gpu_server.py': '"""Persistent GPU job server (runs inside the Kaggle venv; reached through an ngrok tunnel).\nUsage: GPU_SERVER_TOKEN=... [GPU_SERVER_ALLOW_EXEC=1] python kaggle_gpu_server.py ROOT OUT PORT\n\nThe model (Qwen2.5-Omni-3B Thinker, fp16) is loaded ONCE into a namespace together with every helper of temporal_lab.py\n(fetch, balanced_items, prepare_video, qwen_inputs, score, media_layout, run_loop, ...). Jobs are Python snippets executed in\nthat namespace by one worker thread (the GPU is single-tenant). A job may set RESULT (JSON-serialisable) and print.\n\nEndpoints (all but /ping need the header X-Token):\n  GET  /ping                         -> {"ok": true}\n  GET  /info                         -> gpu / memory / queue state\n  POST /submit  {code, name}         -> {job_id}           (needs ALLOW_EXEC)\n  GET  /job/<id>?since=<n>           -> status, new stdout lines, result/error\n  POST /upload  {path, b64}          -> writes OUT/uploads/<path>\n  GET  /file?path=<rel>              -> bytes of OUT/<rel>\n  POST /restart                      -> the process exits; the notebook restarts it (frees a hung GPU)\nThe server only listens on 127.0.0.1; the tunnel forwards to it."""\nimport os, sys, json, time, threading, traceback, io, contextlib, base64, hmac, uuid, queue\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nfrom urllib.parse import urlparse, parse_qs\n\nROOT, OUT, PORT = Path(sys.argv[1]), Path(sys.argv[2]), int(sys.argv[3])\nTOKEN = os.environ[\'GPU_SERVER_TOKEN\']\nALLOW_EXEC = os.environ.get(\'GPU_SERVER_ALLOW_EXEC\', \'1\') == \'1\'\nUP = OUT / \'uploads\'; UP.mkdir(parents=True, exist_ok=True)\nsys.path.insert(0, str(UP)); sys.path.insert(0, str(ROOT))\nos.environ[\'LAB_ROOT\'] = str(ROOT); os.environ[\'LAB_OUT\'] = str(OUT)\n\n# ---- namespace with the model and all helpers ------------------------------------------------------------------------\nNS = {\'__name__\': \'labns\', \'__file__\': str(ROOT / \'temporal_lab.py\')}\nexec(compile((ROOT / \'temporal_lab.py\').read_text(encoding=\'utf-8\'), \'temporal_lab.py\', \'exec\'), NS)\nt0 = time.time()\nNS[\'MODEL_\'], NS[\'PROC\'] = NS[\'load_model\']()\nNS[\'L4\'] = [NS[\'PROC\'].tokenizer.convert_tokens_to_ids(L) for L in \'ABCD\']\nprint(f\'model loaded in {time.time() - t0:.0f}s\', flush=True)\n\nJOBS, Q, LOCK = {}, queue.Queue(), threading.Lock()\n\n\nclass Tee(io.TextIOBase):\n    def __init__(self, job): self.job = job\n    def write(self, s):\n        with LOCK: self.job[\'out\'] += s\n        sys.__stdout__.write(s); return len(s)\n    def flush(self): sys.__stdout__.flush()\n\n\ndef worker():\n    import torch\n    while True:\n        jid = Q.get(); job = JOBS[jid]; job[\'status\'] = \'running\'; job[\'t_start\'] = time.time()\n        NS[\'RESULT\'] = None\n        try:\n            with contextlib.redirect_stdout(Tee(job)):\n                exec(compile(job[\'code\'], f\'<job {job["name"]}>\', \'exec\'), NS)\n            blob = json.dumps(NS.get(\'RESULT\'), default=str)\n            job[\'result\'] = json.loads(blob) if len(blob) < 5_000_000 else {\'_too_large\': len(blob), \'hint\': \'write it to a file and GET /file\'}\n            job[\'status\'] = \'done\'\n        except BaseException:\n            job[\'error\'] = traceback.format_exc(); job[\'status\'] = \'error\'\n        finally:\n            try: torch.cuda.empty_cache()\n            except Exception: pass\n            job[\'t_end\'] = time.time()\n\n\nthreading.Thread(target=worker, daemon=True).start()\n\n\nclass H(BaseHTTPRequestHandler):\n    def log_message(self, *a): pass\n\n    def _send(self, code, obj=None, raw=None, ctype=\'application/json\'):\n        body = raw if raw is not None else json.dumps(obj, default=str).encode()\n        self.send_response(code); self.send_header(\'Content-Type\', ctype); self.send_header(\'Content-Length\', str(len(body))); self.end_headers(); self.wfile.write(body)\n\n    def _auth(self): return hmac.compare_digest(self.headers.get(\'X-Token\', \'\'), TOKEN)\n\n    def _body(self):\n        n = int(self.headers.get(\'Content-Length\', 0)); return json.loads(self.rfile.read(n) or b\'{}\')\n\n    def do_GET(self):\n        u = urlparse(self.path)\n        if u.path == \'/ping\': return self._send(200, {\'ok\': True})\n        if not self._auth(): return self._send(401, {\'error\': \'bad token\'})\n        if u.path == \'/info\':\n            import torch\n            return self._send(200, {\'gpus\': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n                                    \'mem_alloc_GiB\': [round(torch.cuda.memory_allocated(i) / 2 ** 30, 2) for i in range(torch.cuda.device_count())],\n                                    \'queue\': Q.qsize(), \'jobs\': len(JOBS), \'allow_exec\': ALLOW_EXEC, \'uptime_s\': round(time.time() - t0)})\n        if u.path.startswith(\'/job/\'):\n            jid = u.path.split(\'/\')[2]; since = int(parse_qs(u.query).get(\'since\', [\'0\'])[0]); j = JOBS.get(jid)\n            if not j: return self._send(404, {\'error\': \'no such job\'})\n            with LOCK: out = j[\'out\']\n            return self._send(200, {\'status\': j[\'status\'], \'name\': j[\'name\'], \'out\': out[since:], \'out_len\': len(out), \'result\': j.get(\'result\'),\n                                    \'error\': j.get(\'error\'), \'seconds\': round((j.get(\'t_end\') or time.time()) - j.get(\'t_start\', time.time()), 1) if j.get(\'t_start\') else 0})\n        if u.path == \'/file\':\n            rel = parse_qs(u.query).get(\'path\', [\'\'])[0]; p = (OUT / rel).resolve()\n            if OUT.resolve() not in p.parents or not p.is_file(): return self._send(404, {\'error\': \'not found\'})\n            return self._send(200, raw=p.read_bytes(), ctype=\'application/octet-stream\')\n        return self._send(404, {\'error\': \'unknown\'})\n\n    def do_POST(self):\n        u = urlparse(self.path)\n        if not self._auth(): return self._send(401, {\'error\': \'bad token\'})\n        b = self._body()\n        if u.path == \'/submit\':\n            if not ALLOW_EXEC: return self._send(403, {\'error\': \'exec disabled on this server\'})\n            jid = uuid.uuid4().hex[:10]; JOBS[jid] = {\'name\': b.get(\'name\', \'job\'), \'code\': b[\'code\'], \'status\': \'queued\', \'out\': \'\'}; Q.put(jid)\n            return self._send(200, {\'job_id\': jid})\n        if u.path == \'/upload\':\n            rel = Path(b[\'path\']); p = (UP / rel).resolve()\n            if UP.resolve() not in p.parents: return self._send(400, {\'error\': \'path must stay under uploads/\'})\n            p.parent.mkdir(parents=True, exist_ok=True); p.write_bytes(base64.b64decode(b[\'b64\'])); return self._send(200, {\'written\': str(rel), \'bytes\': p.stat().st_size})\n        if u.path == \'/restart\':\n            self._send(200, {\'restarting\': True}); threading.Thread(target=lambda: (time.sleep(0.5), os._exit(3)), daemon=True).start(); return\n        return self._send(404, {\'error\': \'unknown\'})\n\n\nprint(f\'serving on 127.0.0.1:{PORT} (allow_exec={ALLOW_EXEC})\', flush=True)\nThreadingHTTPServer((\'127.0.0.1\', PORT), H).serve_forever()\n'}
for _n, _s in SERVER_FILES.items(): (ROOT / _n).write_text(_s, encoding='utf-8')
print('Đã ghi:', list(SERVER_FILES))


In [ ]:
# CELL 3 — khởi động server + ngrok, giữ notebook sống
ALLOW_EXEC = True      # False = chỉ kiểm tra kết nối, server không nhận mã
PORT = 8765
import secrets, subprocess, time, sys, os, urllib.request
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], check=True)
from pyngrok import ngrok

def ngrok_token():
    t = os.environ.get('NGROK_AUTHTOKEN', '')
    if t: return t
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('NGROK_AUTHTOKEN')
    except Exception as e:
        raise RuntimeError('Không đọc được Kaggle Secret NGROK_AUTHTOKEN (' + type(e).__name__ + '). Add-ons -> Secrets -> thêm NGROK_AUTHTOKEN và bật Attach.')

TOKEN = secrets.token_urlsafe(32)
env = {**os.environ, 'GPU_SERVER_TOKEN': TOKEN, 'GPU_SERVER_ALLOW_EXEC': '1' if ALLOW_EXEC else '0', 'PYTHONUNBUFFERED': '1'}
cmd = [str(PYTHON), str(ROOT / 'kaggle_gpu_server.py'), str(ROOT), str(OUT), str(PORT)]
SERVER_LOG = OUT / 'server.log'

def start_server():
    return subprocess.Popen(cmd, stdout=open(SERVER_LOG, 'a', encoding='utf-8'), stderr=subprocess.STDOUT, env=env)

def wait_ready(timeout=1200):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{PORT}/ping', timeout=3); return True
        except Exception:
            time.sleep(5)
    return False

server = start_server()
print('Đang nạp model (3–6 phút lần đầu)...', flush=True)
if not wait_ready():
    print(SERVER_LOG.read_text()[-3000:]); raise RuntimeError('server không lên được; xem server.log ở trên')
ngrok.set_auth_token(ngrok_token())
public_url = ngrok.connect(PORT, 'http')
print('=' * 70)
print('  URL   :', public_url)
print('  TOKEN :', TOKEN)
print('=' * 70, flush=True)

while True:                       # giữ notebook sống; tự khởi động lại server khi /restart hoặc khi nó chết
    time.sleep(30)
    if server.poll() is not None:
        print('server dừng (mã', server.returncode, ') -> khởi động lại', flush=True)
        server = start_server()
        if not wait_ready(): print('khởi động lại thất bại; xem server.log', flush=True)
